# 01 — PDF Ingestion & English Extraction

**Purpose:** Explore the raw PDF, understand the page structure,
and extract only English-language question pages.



---

## 1. Setup

In [ ]:
import sys
sys.path.append('..')   # so we can import from src/ later

import pdfplumber
import re
import json
from pathlib import Path

# Paths
RAW_DIR   = Path('../data/raw')
OUT_DIR   = Path('../data/processed')
OUT_DIR.mkdir(exist_ok=True)

print('✅ Setup done')

## 2. Explore a single PDF — check page count & raw text

In [ ]:
PDF_PATH = RAW_DIR / '30-1-2_Mathematics_Standard.pdf'

with pdfplumber.open(PDF_PATH) as pdf:
    print(f'Total pages: {len(pdf.pages)}')
    print()
    # Look at first 5 pages raw
    for i, page in enumerate(pdf.pages[:5]):
        text = page.extract_text() or ''
        print(f'--- PAGE {i+1} (first 200 chars) ---')
        print(text[:200])
        print()

## 3. Identify Hindi vs English pages

In [ ]:
def is_english_page(text: str) -> bool:
    """Returns True if page contains English question content."""
    english_patterns = [
        r'\d+\.\s+[A-Z][a-z]',
        r'(Find|Prove|If |The |A |An |SECTION|Case Study|Determine|Show that)',
    ]
    hindi_pattern = r'[\u0900-\u097F]'   # Devanagari unicode block

    has_hindi   = bool(re.search(hindi_pattern, text))
    has_english = any(re.search(p, text) for p in english_patterns)

    return has_english and not has_hindi


# Map every page
with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ''
        label = '✅ ENGLISH' if is_english_page(text) else '⛔ SKIP'
        print(f'Page {i+1:2d}: {label}  | {text[:60].strip()}')

## 4. Extract all English pages from one PDF

In [ ]:
def extract_english_pages(pdf_path: Path, paper_name: str, year: str) -> list:
    """Extract only English question pages from a CBSE bilingual PDF."""
    results = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if not text:
                continue
            text = text.strip()
            if is_english_page(text):
                results.append({
                    'page_num':  i + 1,
                    'paper':     paper_name,
                    'year':      year,
                    'text':      text
                })
    return results


pages = extract_english_pages(
    PDF_PATH,
    paper_name='Mathematics_Standard_SET2',
    year='2024'
)

print(f'Extracted {len(pages)} English pages')
print()
for p in pages:
    print(f"  Page {p['page_num']:2d}: {p['text'][:70]}")

## 5. Run on all PDFs in data/raw/ and save

In [ ]:
# Auto-discover all PDFs in data/raw/
pdf_files = list(RAW_DIR.glob('*.pdf'))
print(f'Found {len(pdf_files)} PDFs: {[f.name for f in pdf_files]}')

all_pages = []
for pdf_path in pdf_files:
    # Derive paper name from filename
    paper_name = pdf_path.stem.replace('-', '_')
    pages = extract_english_pages(pdf_path, paper_name=paper_name, year='2024')
    all_pages.extend(pages)
    print(f'  {pdf_path.name} → {len(pages)} pages')

# Save
out_path = OUT_DIR / 'raw_english_pages.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(all_pages, f, indent=2, ensure_ascii=False)

print(f'\n✅ Saved {len(all_pages)} pages → {out_path}')